**TRAINING CONVOLUTIONAL AUTOENCODERS 1D**

In [ ]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

**SETUP PARAMETERS**

In [ ]:
job_name = "check_sar_v1"                                    
dataset = "Test" 
sensor = "SAR"

n_channels = 2 if sensor == "SAR" else 10 # OPT
                   
parametri = {
    "job_name": job_name,
    "dataset": dataset,
    "sensor": sensor,
    "resume": False,
    "time_debug": False,

    "epochs": 5, 
    "batch_size": 16, 
    "lr": 1e-4, 
    "weight_decay": 1e-4,      
    "patch_size": 256, 
    "n_images": 4,                       
    "n_channels": n_channels,                     
    "latent_space_vector_dim": 512,                                                                                           
    "workers": 0,
    "dataset": dataset,                                
    "patience": 10,                                    
    "min_delta": 1e-4,                       
}

volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "200Gi"}   
    }
]

**BUILD ENVIRONMENT**

In [ ]:
autoencoder_1D_train_func = project.new_function(
    name= f'autoencoder_1D_{job_name}',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="train_autoencoder_1D", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30", "torch==2.1.2", "matplotlib==3.10.9", "digitalhub==0.15.11", "digitalhub-runtime-python==0.15.2"]
)

build = autoencoder_1D_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")

**TRAINING**

In [ ]:
run_autoencoder_1D = autoencoder_1D_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xrtxa5000-shared",                                       
    local_execution= False,
    wait=True                               
)

print(f"Run train_encoders avviato: {run_autoencoder_1D.id}")

**PLOTS**

In [ ]:
path = project.get_artifact(f"autoencoder1D_{sensor}_metrics_{job_name}").download(overwrite=True)
df = pd.read_csv(path)

plt.figure(figsize=(8, 5))
plt.plot(df['epoch'], df['train_loss'], color='blue', label='Train Loss')
plt.title(f'CAE {sensor} 1D - {job_name}')
plt.xlabel('Epochs')
plt.ylabel('Loss MSE')
plt.grid(True)
plt.legend()
plt.show()